# Adım 2: Veri Temizleme ve Ön İşleme (Data Cleaning & Preprocessing)

Bu not defteri, Öğrenci Performansı veri setinin (**student_performance.csv**) Makine Öğrenmesi (Machine Learning) algoritmaları için temizlenmesini, optimize edilmesini ve hazırlanmasını sağlamaktadır.
Yönetici sunumlarına ve kalite standartlarına uygun olarak veri seti kalitesi, veri kayıpları ve uygulanan stratejiler detaylandırılmıştır.

## 🎯 Temel Hedefler ve Yapılan İşlemler
1. **Eksik Veri (Missing Data) Yönetimi:** Eksik verilerin tespit edilmesi ve genel dağılımı (**%0 Veri Kaybı**) korumak adına medyan (ortanca) değerler ile doldurulması.
2. **Veri Tiplerinin ve Bellek Optimizasyonunun Sağlanması:** Kategorik eğitim seviyesinin (`Parent_Education_Level`) `category` analitik tipine dönüştürülmesi ve ID'lerin veri formatından ayrıştırılması.
3. **Sayısal Hassasiyet (Decimal) ve Mantıksal Bütünlük:** 
   - `Stress_Level` değişkeninin (1-10 arası) int (tam sayı) yapılması.
   - Notlar, mesafeler ve oranların 2 ondalık basamağa (örn: `62.74`); saat ve zaman dilimlerinin 1 ondalık basamağa (örn: `8.5`) yuvarlanarak gürültüden (noise) arındırılması.
4. **Aykırı ve Tutarsızlık (Outliers/Inconsistencies) Çözümü:** İmkansız değerlerin (örn. günlük 150 saat uyku süresi, negatif / eksi okul mesafesi) tespit edilip mantıklı sınırlara (örn: 18 saat uyku medyanı, mutlak değer mesafe) çekilmesi.
5. **Mükerrer (Duplicate) Kopya Kontrolü:** Ekstra veya tekrarlayan hatalı satırların temizlenmesi.

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
print("Gerekli veri bilimi kütüphaneleri başarıyla yüklendi.")

Gerekli veri bilimi kütüphaneleri başarıyla yüklendi.


## 1. Veri Okuma ve İlk Durum Raporu

Ham veri setini ekleyerek, kaç satır/sütundan oluştuğunu inceliyoruz.

In [2]:
df = pd.read_csv('dataset/student_performance.csv')

total_records = len(df)
print(f"Toplam Kayıt Sayısı: {total_records}")
print(f"Toplam Özellik (Sütun) Sayısı: {df.shape[1]}")

Toplam Kayıt Sayısı: 10000
Toplam Özellik (Sütun) Sayısı: 12


## 2. Eksik Veri Analizi ve Doldurma (Imputation)

**Mevcut Durum:** Veri setinde `Weekly_Study_Hours`, `Internet_Usage_Time` ve `Extracurricular_Activities` metriklerinde bazı öğrencilere ait veriler eksiktir (`NaN`). 
**Uygulanan Strateji:** Bu eksiklikler toplam veri setinin maksimum **%4.2**'sini oluşturduğu için satırların silinmesi yerine veri eksenini en az saptıran **Medyan (Ortanca)** değerler ile doldurma işlemi (*Imputation*) yapılmıştır.

In [3]:
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]

print("--- Temizlik Öncesi Eksik Veri Özeti ---")
for col in missing_data.index:
    perc = (missing_data[col] / total_records) * 100
    print(f"- {col}: {missing_data[col]} adet eksik kayıt (Tüm verinin %{perc:.2f}'si)")

# Eksik verilerin Medyan ile onarılması
for col in missing_data.index:
    df[col] = df[col].fillna(df[col].median())

print("\n=> Başarılı: Tüm eksik veriler ilgili kolonun 'Medyan' değerleriyle dolduruldu.")
print(f"=> Kalan eksik satır sayısı: {df.isnull().sum().sum()} (Kayıp: %0.00)")

--- Temizlik Öncesi Eksik Veri Özeti ---
- Weekly_Study_Hours: 312 adet eksik kayıt (Tüm verinin %3.12'si)
- Internet_Usage_Time: 427 adet eksik kayıt (Tüm verinin %4.27'si)
- Extracurricular_Activities: 256 adet eksik kayıt (Tüm verinin %2.56'si)

=> Başarılı: Tüm eksik veriler ilgili kolonun 'Medyan' değerleriyle dolduruldu.
=> Kalan eksik satır sayısı: 0 (Kayıp: %0.00)


## 3. Optimizasyon: Veri Tipi Optimizasyonu ve Dönüşümleri

Modellemenin başarısı ve algoritma sağlığı adına:
- **`Parent_Education_Level`:** Algoritmanın seviyeleri kolay ayırması için `category` (Kategorik) formata alındı.
- **`Student_ID`:** İstatistiksel yanılsama yaratmaması için matematiksel işlemden düşürülüp metinsel `string / object` tipe alındı.

In [4]:
df['Student_ID'] = df['Student_ID'].astype(str)
df['Parent_Education_Level'] = df['Parent_Education_Level'].astype('category')

print("=> Dönüşüm İşlemi Tamamlandı: Bellek optimizasyonu yapıldı.\n")
print(df[['Student_ID', 'Parent_Education_Level']].dtypes)

=> Dönüşüm İşlemi Tamamlandı: Bellek optimizasyonu yapıldı.

Student_ID                  object
Parent_Education_Level    category
dtype: object


## 4. Tutarsız ve Aykırı Değerlerin (Outliers) Giderilmesi

Veri tabanındaki hatalı girişler analiz edilerek mantıksal boyuta çekildi.

**Tespit Edilen Sorunlar:**
- **Negatif (Eksi) Okul Mesafesi:** Ev ile okul aralığı negatif (`-5.0`) değer alamaz. Veri yönelim hatası tespit edilerek, rakamlar **Mutlak Değer** çerçevesinde pozitife oturtuldu.
- **İmkansız Uyku Saatleri:** Örneğin günlük uyku süresinin 24 saati geçtiği (Örn: 150 saat) tutarsız satırlar (%1.43) bulunmuştur. Prensip olarak mantıksal bir **üst sınır (18 saat)** uygulanmış, bu sınırı geçen hatalı durumlar veri dağılımını bozmayan normal medyan uyku değeri ile değiştirilmiştir.

In [5]:
# Mesafe - Ortak bir düzlem için Mutlak değer kullanıyoruz
neg_dist = (df['Distance_to_School'] < 0).sum()
dist_perc = (neg_dist / total_records) * 100
df['Distance_to_School'] = df['Distance_to_School'].abs()

# Uyku Süresi - Üst limitten mantıklı olan medyan rakamıyla onarım
upper_sleep_limit = 18.0
abnormal_sleep = (df['Sleep_Hours'] > upper_sleep_limit).sum()
abnormal_perc = (abnormal_sleep / total_records) * 100

sane_median = df.loc[df['Sleep_Hours'] <= upper_sleep_limit, 'Sleep_Hours'].median()
df.loc[df['Sleep_Hours'] > upper_sleep_limit, 'Sleep_Hours'] = sane_median

print("--- Tutarsız Veri Müdahalesi Analizi ---")
print(f"- Tespit edilen Negatif Mesafe Değerleri: Bulunan {neg_dist} adet kayıt (Tüm verinin %{dist_perc:.2f}'si) 'Mutlak Değer' kuralıyla onarıldı.")
print(f"- Aşırı Uyku Süresi Sınır İhlali (> 18 saat): Bulunan {abnormal_sleep} adet hatalı kayıt (Tüm verinin %{abnormal_perc:.2f}'si), sağlıklı medyan uyku süresi ({sane_median:.2f} saat) ile değiştirildi.")

--- Tutarsız Veri Müdahalesi Analizi ---
- Tespit edilen Negatif Mesafe Değerleri: Bulunan 218 adet kayıt (Tüm verinin %2.18'si) 'Mutlak Değer' kuralıyla onarıldı.
- Aşırı Uyku Süresi Sınır İhlali (> 18 saat): Bulunan 143 adet hatalı kayıt (Tüm verinin %1.43'si), sağlıklı medyan uyku süresi (8.41 saat) ile değiştirildi.


## 5. Mükerrer (Duplicate) Kopya Sistem Kaydı Kontrolü

Aynı öğrencinin çift kopya ile makine öğrenmesini ve algoritma tarafsızlığını (bias) etkilememesi adına tekrarlanan veri kontrolü yapılır.

In [7]:
dup_count = df.duplicated().sum()
dup_perc = (dup_count / total_records) * 100

if dup_count > 0:
    df = df.drop_duplicates()
    print(f"Uyari: Sistemde {dup_count} adet (%{dup_perc:.2f}) tekrarlanan (kopya) öğrenci verisi bulundu ve filtrelendi.")
else:
    print("Veri Seti Onayı: Tekrarlayan (mükerrer) sistem içi kayıt bulunmamaktadır (%0.00).")

Veri Seti Onayı: Tekrarlayan (mükerrer) sistem içi kayıt bulunmamaktadır (%0.00).


## 6. Sayısal Verilerde Mantıksal Hassasiyet (Decimal) ve Tam Sayı (Integer) Düzeltmesi

Gereksiz virgül (decimal) ondalık basamakları performansı yavaşlatır ve gürültü yaratır. Bu sebeple veride mantıksal yuvarlamalar (rounding) sağlandı:
- **`Stress_Level`:** Stres gibi psikolojik ölçekler **Tam Sayı** olur, buküsuratlar elenerek `integer` sisteme geçirildi.
- **1 Ondalık İhtiyacı Olanlar (Süreler vb.):** `Weekly_Study_Hours`, `Sleep_Hours`, `Internet_Usage_Time`, `Extracurricular_Activities` metrikleri ölçü birimi gereği (Örn: 8.5 Saat) **1 ondalık basamağa (`.round(1)`)** sabitlendi.
- **2 Ondalık İhtiyacı Olanlar (Skor, Mesafe vb.):** `Attendance_Rate`, `Previous_Scores`, `Distance_to_School`, `Exam_Score` yapılarının detayını (skorunu) yitirmemesi adına **2 ondalık basamağa (`.round(2)`)** sabitlendi.

In [8]:
# Stres Seviyesini küsurattan arındırarak tam sayı (Int) formuna aldık.
df['Stress_Level'] = df['Stress_Level'].round().astype(int)

# 1 Ondalık (Decimal) Basamağa Yuvarlanan Zaman Metrikleri
cols_1_dec = ['Weekly_Study_Hours', 'Sleep_Hours', 'Internet_Usage_Time', 'Extracurricular_Activities']
df[cols_1_dec] = df[cols_1_dec].round(1)

# 2 Ondalık (Decimal) Basamağa Yuvarlanan Skor ve Puanlar
cols_2_dec = ['Attendance_Rate', 'Previous_Scores', 'Distance_to_School', 'Exam_Score']
df[cols_2_dec] = df[cols_2_dec].round(2)

print("Bütün sayısal verilerin virgül problemleri temizlendi ve tip optimizasyon işlemi başarılı oldu.\n")
display(df[['Student_ID', 'Stress_Level'] + cols_1_dec + cols_2_dec].head())

Bütün sayısal verilerin virgül problemleri temizlendi ve tip optimizasyon işlemi başarılı oldu.



,Student_ID,Stress_Level,Weekly_Study_Hours,Sleep_Hours,Internet_Usage_Time,Extracurricular_Activities,Attendance_Rate,Previous_Scores,Distance_to_School,Exam_Score
0,10000,6,28.5,6.9,8.2,5.8,92.43,88.87,2.02,62.74
1,10001,1,18.2,8.8,8.0,19.2,84.94,53.51,35.22,54.92
2,10002,3,20.0,9.6,8.6,15.6,91.22,32.50,12.94,42.86
3,10003,5,20.0,7.4,6.9,1.5,94.81,63.86,5.59,63.47
4,10004,4,23.8,8.4,7.2,19.1,83.93,59.74,19.00,57.80


## 7. Temizlenmiş Verinin Dışa Aktarımı (%100 Clean Export)

Veri seti artık tüm pürüzlerden, boş verilerden ve gürültü/aykırı tutarsızlıklardan (`outlier/noise`) tamamen arındırılmıştır. 
Yeni temiz veritabanımız Modelleme (Makine Öğrenimi) aşamasına başlamak üzere risksiz olarak `student_performance_cleaned.csv` konumuna CSV olarak teslimat için dışa aktarılıyor.

In [9]:
output_path = 'dataset/student_performance_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\n🏆 Yüksek kalitede modellenmeye hazır hale getirilen veri seti '{output_path}' konumuna kaydedildi.\n")

print("--- Temizlenmiş Pipeline Sonrası İstatistiksel Durum ---")
display(df.describe(include='all'))


🏆 Yüksek kalitede modellenmeye hazır hale getirilen veri seti 'dataset/student_performance_cleaned.csv' konumuna kaydedildi.

--- Temizlenmiş Pipeline Sonrası İstatistiksel Durum ---


,Student_ID,Weekly_Study_Hours,Attendance_Rate,Sleep_Hours,Previous_Scores,Internet_Usage_Time,Parent_Education_Level,Distance_to_School,Stress_Level,Practice_Exams_Passed,Extracurricular_Activities,Exam_Score
count,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000,10000.00000,10000.000000,10000.00000,10000.000000,10000.000000
unique,10000,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN
top,10000,NaN,NaN,NaN,NaN,NaN,Yok,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,2047,NaN,NaN,NaN,NaN,NaN
mean,NaN,19.964800,86.931816,8.404640,59.752330,7.542900,NaN,19.91071,3.166200,9.49770,9.926480,53.680732
std,NaN,5.799928,5.423626,0.675854,16.813003,1.073283,NaN,11.47534,1.356747,2.75876,5.697857,10.541100
min,NaN,2.000000,74.130000,5.800000,24.010000,2.700000,NaN,0.50000,1.000000,0.00000,0.000000,21.840000
25%,NaN,16.000000,82.920000,8.000000,45.910000,6.800000,NaN,9.76000,2.000000,8.00000,5.000000,46.340000
50%,NaN,20.000000,86.905000,8.400000,59.930000,7.600000,NaN,19.76000,3.000000,9.00000,10.000000,53.640000
75%,NaN,23.900000,90.992500,8.900000,73.392500,8.400000,NaN,29.97250,4.000000,11.00000,14.700000,61.002500
